# Interpreta un registro de optimización operacional

Nivel: 1. Requisitos previos: Nivel 2 de conminución y nivel 1 de aprendizaje. Tiempo de lectura: 18 minutos.

## Objetivos

Distingue las decisiones, el objetivo, las restricciones de factibilidad y el esfuerzo de búsqueda medido en el resultado comprometido. Explica por qué el óptimo depende del caso de autoría propia y su contrato.

## Preparación

```bash
pip install numpy==2.2.6 scipy==1.17.1 pandas==2.2.3 matplotlib==3.11.2 nbclient==0.11.0 nbformat==5.11.1 ipykernel==7.4.0 SALib==1.6.0
```

El pipeline OreFlow se importa desde `data-pipeline/`; este notebook nunca instala paquetes.

Configura las importaciones y registra versiones. Este paso hace reproducible la ejecución del método.

In [1]:
# Importa herramientas de análisis y registra sus versiones exactas.
import importlib.metadata
import json
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Resuelve el pipeline local desde la estructura relativa del repositorio.
repo = Path('../../../').resolve()
sys.path.insert(0, str(repo / 'data-pipeline'))
for package in ('numpy', 'scipy', 'pandas', 'matplotlib', 'nbclient', 'nbformat', 'ipykernel'):
    print(package, importlib.metadata.version(package))

numpy 2.2.6
scipy 1.17.1
pandas 2.2.3
matplotlib 3.11.2
nbclient 0.11.0
nbformat 5.11.1
ipykernel 7.4.0


## Datos

Responsable: autores de OreFlow; `data/derived/benchmark.json` y `data/derived/contract/operating_contract.json`, licencia MIT del proyecto. El óptimo se calcula para un escenario propio, no para una planta. Repositorio: https://github.com/fsantibanezleal/CAOS_OreFlow.

## Método

OreFlow maximiza el metal primario recuperado respecto del punto base, respetando límites de ley del concentrado, potencia instalada y agua. Una búsqueda generalizada por patrones opera en el cubo unitario de los límites declarados. Un candidato rechazado por contrato o motor nunca se considera factible. El registro muestra decisiones, restricciones activas, evaluaciones y trayectoria de ponderaciones.

Carga el registro exacto del benchmark y muestra las decisiones desde la base hasta el óptimo junto con la evidencia de factibilidad.

In [2]:
import json
from pathlib import Path
import pandas as pd
from pipeline.cases.catalog import CASES

# Paso ejecutable del método OreFlow explicado en la celda anterior.
repo = Path('../../../').resolve()
benchmark = json.loads((repo / 'data/derived/benchmark.json').read_text(encoding='utf-8'))
record = benchmark['optimization']['copper_porphyry_soft']['nominal']
case = next(item for item in CASES if item.id == 'copper_porphyry_soft')
case_record = json.loads((repo / 'data/derived/cases/copper_porphyry_soft.json').read_text(encoding='utf-8'))
base_point = case_record['variants'][0]['point']
print('status:', record['status'], '| algorithm: generalized pattern search')
print('base feasible:', record['base_feasible'], '| screened:', record['screened'], '| evaluations:', record['evaluations'])
rows = [{'state': 'base', **{name: base_point[name] for name in record['decisions']}}, {'state': 'optimum', **record['decisions']} ]
display(pd.DataFrame(rows).set_index('state').T)

status: optimal | algorithm: generalized pattern search
base feasible: True | screened: True | evaluations: 309


state,base,optimum
target_p80_um,150.0,124.291992
collector_gpt,25.0,74.951172
jg_cm_s,1.4,2.499609


Compara la mejora del objetivo con la restricción física activa. El objetivo del optimizador no incluye precio del mineral ni costo operacional.

In [3]:
# Compara la mejora del objetivo con la restricción física activa.
print('recovered metal gain (%):', round(record['gain_pct'], 3))
print('active constraints:', record['active'])
print('same optimum without surrogate screen:', record['same_optimum_without_screen'])
print('screened engine evaluations:', record['evaluations'])
print('unscreened engine evaluations:', record['evaluations_without_screen'])
print('contract decision bounds:')
inputs = json.loads((repo / 'data/derived/contract/operating_contract.json').read_text(encoding='utf-8'))['cases']['copper_porphyry_soft']['inputs']
display(pd.DataFrame([{'decision': name, 'min': inputs[name]['min'], 'max': inputs[name]['max'], 'unit': inputs[name]['unit']} for name in record['decisions']]).set_index('decision'))

recovered metal gain (%): 5.744
active constraints: ['power']
same optimum without surrogate screen: True
screened engine evaluations: 309
unscreened engine evaluations: 266
contract decision bounds:


,min,max,unit
decision,,,
target_p80_um,75.0,300.0,um
collector_gpt,0.0,75.0,g/t
jg_cm_s,0.5,2.5,cm/s


## Resultados

El registro informa un candidato factible y una mejora medida bajo el objetivo y las restricciones declarados. Coincidir con la búsqueda sin filtro es una comprobación de paridad; no prueba optimalidad global ni valor económico. Los datos y capacidades de planta son supuestos del escenario.

## Resumen

Interpretaste el optimizador como un cálculo de ingeniería restringido: depende del objetivo, límites, restricciones y caso de partida. Un resultado factible no es una recomendación operacional.

## Referencias

- https://doi.org/10.1137/S1052623493250780
- https://doi.org/10.1137/070692662